# A1 · Ejercicio de embeddings manuales
**LSCA2318 · Tema 2.** Antes de que un modelo calcule vectores, tú los vas a construir a mano para entender qué significa «parecido = cerca».

**Qué haces:**
1. Eliges **4 a 6 dimensiones con nombre** que tengan sentido en tu variante (ej. `seguridad, pruebas, dinero, despliegue`).
2. Escribes **3 frases** de tu variante: F1 y F2 del mismo tema, F3 de otro tema.
3. Les das un valor de 0 a 1 a cada dimensión (justifícalo en una línea).
4. Normalizas y calculas el coseno F1–F2 y F1–F3. **Interpretas** el resultado.

Funciona en Colab o en VS Code. No necesita internet ni llave.

In [ ]:
import numpy as np

# 1) Dimensiones con nombre (variante v4-documentador)
dimensiones = ['seguridad', 'errores', 'ejemplos', 'estilo', 'nombres']

# 2) y 3) Frases del corpus de mi variante y sus vectores (0 a 1 por dimensión, en el mismo orden).
frases = {
    'F1': ('No copies la llave que está en el código dentro del README; menciónala solo como la variable de entorno PAGOS_API_KEY.',
        [0.9, 0.0, 0.1, 0.2, 0.2]),
    'F2': ('En la documentación para socios no se publican las rutas /internal ni los secretos del servicio.',
        [0.8, 0.0, 0.0, 0.1, 0.3]),
    'F3': ('Cada error HTTP documentado debe decir cuándo ocurre y qué hacer, con un ejemplo de respuesta.',
        [0.1, 0.9, 0.7, 0.2, 0.0]),
}
for k, (texto, v) in frases.items():
    assert len(v) == len(dimensiones), f"{k} debe tener {len(dimensiones)} valores"
    print(k, texto, dict(zip(dimensiones, v)))

**Justificación de tus valores:**

- F1 (seguridad 0.9): habla de no copiar una llave al README, que es la regla de manejo de secretos (doc-05). Tiene un poco de «ejemplos» (0.1), «estilo» (0.2) y «nombres» (0.2) porque menciona una variable de entorno con nombre propio.
- F2 (seguridad 0.8): trata el mismo tema que F1, qué no se publica (secretos y rutas /internal). Pesa algo menos en seguridad porque también habla de rutas (nombres 0.3).
- F3 (errores 0.9, ejemplos 0.7): trata cómo documentar errores HTTP y pide un ejemplo de respuesta (doc-06). Casi nada de seguridad (0.1).

In [ ]:
def normalizar(v):
    v = np.asarray(v, dtype=float)
    n = np.linalg.norm(v)
    return v / n if n > 0 else v      # el vector cero se queda en cero

def coseno(a, b):
    a, b = normalizar(a), normalizar(b)
    return float(a @ b)

V = {k: v for k, (_, v) in frases.items()}
for k in V:
    print(f"{k} normalizado:", np.round(normalizar(V[k]), 3), " norma =", round(np.linalg.norm(normalizar(V[k])), 3))

print(f"\ncos(F1, F2) = {coseno(V['F1'], V['F2']):.3f}   ← mismo tema")
print(f"cos(F1, F3) = {coseno(V['F1'], V['F3']):.3f}   ← temas distintos")

**Interpretación:**

1. Sí: cos(F1, F2) = 0.980 es mucho mayor que cos(F1, F3) = 0.181. F1 y F2 hablan del mismo tema (qué no se publica por seguridad) y sus vectores apuntan casi en la misma dirección. F3 trata de errores, así que casi no comparte dimensiones con F1.
2. La dimensión «seguridad» pesó más: es la que concentra el valor alto en F1 y F2 y la que casi vale 0 en F3. El coseno mide la dirección del vector, no su tamaño.
3. Con cientos de dimensiones sin nombre, un modelo capta matices que yo no definí, como sinónimos y contexto. Se pierde interpretabilidad: ya no se puede decir qué significa cada dimensión ni explicar por qué dos textos salieron parecidos. Mis cinco dimensiones sí se explican, pero solo cubren lo que yo imaginé.